# Training YOLO — Deteksi Parkir (3 kelas)
Kelas: `space-empty`, `space-occupied`, `illegal-parking`.
Dataset sudah di-merge & diseragamkan ke bbox oleh `scripts/merge_yolo_datasets.py`.

> Catatan: kelas `illegal-parking` sangat sedikit (~273 box). Cell terakhir memakai augmentasi agresif; pantau recall kelas illegal.

In [1]:
# 1. Install (skip jika sudah ada)
%pip install -q ultralytics

^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# 2. Setup: temukan data.yaml & pilih device otomatis
from pathlib import Path
import torch

# ROOT = folder project (parent dari notebooks/). Sesuaikan bila jalan di Colab.
ROOT = Path.cwd()
if not (ROOT / "dataset_merged").exists():
    ROOT = ROOT.parent  # jalan dari dalam notebooks/
DATA = ROOT / "dataset_merged" / "data.yaml"
assert DATA.exists(), f"data.yaml tidak ditemukan: {DATA}"

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("data  :", DATA)
print("device:", DEVICE, "(" + (torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU") + ")")

data  : c:\Users\LENOVO\Smart-EvaluationSensing-Statistical-System\dataset_merged\data.yaml
device: cpu (CPU)


In [2]:
# 3. Train
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATA),
    epochs=15,
    imgsz=320,
    batch=8,
    device="cpu",
    patience=8,
    cache=False,
    workers=4,
    project=str(ROOT / "runs"),
    name="parking_yolov8n_cpu",
    mosaic=0.0, mixup=0.0, copy_paste=0.0,
)

Ultralytics 8.4.142  Python-3.10.11 torch-2.14.0+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\LENOVO\Smart-EvaluationSensing-Statistical-System\dataset_merged\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=0.0,

In [3]:
# 4. Evaluasi di test split (per-kelas)
best = YOLO(results.save_dir / "weights" / "best.pt")   # FIX: pakai / bukan +
metrics = best.val(data=str(DATA), split="test", device=DEVICE)
print("mAP50-95:", metrics.box.map)
print("mAP50   :", metrics.box.map50)
for i, name in metrics.names.items():
    print(f"  {name:16} AP50={metrics.box.ap50[i]:.3f}")

Ultralytics 8.4.142  Python-3.10.11 torch-2.14.0+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
Model summary (fused): 72 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.40.1 ms, read: 1.41.5 MB/s, size: 20.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\Users\LENOVO\Smart-EvaluationSensing-Statistical-System\dataset_merged\test\labels... 658 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 658/658 269.0it/s 2.4s0.1s
val: New cache created: C:\Users\LENOVO\Smart-EvaluationSensing-Statistical-System\dataset_merged\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 42/42 1.6it/s 26.9s0.6ss
                   all        658      34493      0.847      0.764      0.774      0.595
           space-empty        532      18100      0.989      0.971   

In [4]:
# 5. Prediksi contoh (visual sanity check)
sample_candidates = sorted((ROOT / "dataset_merged" / "test" / "images").glob("*.jpg"))
assert sample_candidates, "Tidak ada file .jpg di folder test/images"
sample = sample_candidates[0]

best.predict(sample, save=True, conf=0.25, device=DEVICE)
print("hasil tersimpan di folder runs/.../predict")


image 1/1 c:\Users\LENOVO\Smart-EvaluationSensing-Statistical-System\dataset_merged\test\images\carpark_vlcsnap-2023-06-21-15h07m52s844_png.rf.9a5237c17d698d74fb1110977b142c39.jpg: 320x320 2 space-occupieds, 84.5ms
Speed: 7.2ms preprocess, 84.5ms inference, 3.4ms postprocess per image at shape (1, 3, 320, 320)
Results saved to C:\Users\LENOVO\Smart-EvaluationSensing-Statistical-System\runs\detect\predict
hasil tersimpan di folder runs/.../predict
